# 钢板缺陷数据的聚类分析（无监督学习）

与 `Faulty_Steel_Plates.ipynb`（有监督分类）使用**同一份数据**，
但换个视角：假装我们不知道标签，只用前 27 个特征探索数据的自然结构。

- **K-Means 聚类**：把钢板按特征相似度分组
- **肘部法 + 轮廓系数**：如何选择簇的数量 k
- **PCA 降维可视化**：把 27 维特征压缩到 2 维作图

> 注意：真实标签（后 7 列独热编码）**不参与任何训练**，
> 只在最后用来评估聚类结果与真实缺陷类型的吻合程度。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

# 中文显示：使用系统自带的 Noto Sans CJK 字体
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
csv_path = Path.cwd() / "faults.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day46-60/faults.csv")

df = pd.read_csv(csv_path)

FEATURES = df.columns[:27]  # 前 27 个特征：聚类唯一用到的输入
LABELS = df.columns[27:]    # 后 7 列：真实类别，仅用于事后评估

X = df[FEATURES]
y = df[LABELS].idxmax(axis=1)  # 还原成单标签，只用于评估

X.head()


## 标准化

K-Means 基于欧氏距离，量纲大的特征（如 `Pixels_Areas` 几十万量级）
会主导距离计算，因此与 KNN/SVM 一样必须先标准化。
注意这里是对**全部数据**做拟合（无监督学习没有训练/测试之分）。


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## K-Means 聚类

真实类别有 7 种，所以先试试 k=7。K-Means 只知道“这是第 0~6 簇”，
簇编号和真实类别名没有任何对应关系，需要用交叉表人工对账。


In [ ]:
kmeans = KMeans(n_clusters=7, n_init="auto", random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# 交叉表：行 = 真实类别，列 = 聚类簇编号
pd.crosstab(y, clusters, rownames=["真实类别"], colnames=["簇编号"])


## 如何选择 k：肘部法 + 轮廓系数

无监督学习没有标签告诉我们 k 该取多少，常用两个启发式：

- **肘部法（Elbow）**：画出簇内平方和（inertia）随 k 的变化，
  找曲线拐点——k 再增大也压缩不了多少簇内距离。
- **轮廓系数（Silhouette）**：衡量样本与本簇的贴合度 vs 与其他簇的分离度，
  取值 [-1, 1]，越大说明聚类结构越清晰。


In [ ]:
ks = range(2, 11)
inertias, silhouettes = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, "o-")
axes[0].set(xlabel="k", ylabel="inertia", title="肘部法")
axes[1].plot(ks, silhouettes, "o-")
axes[1].set(xlabel="k", ylabel="silhouette", title="轮廓系数")
plt.tight_layout()
plt.show()


## 用真实标签评估聚类效果

虽然训练没用标签，但既然我们有，就可以量化“聚类结构”与“真实类别”
的一致程度（注意：这与分类的准确率不是同一个概念）：

- **ARI（调整兰德指数）**：取值 [-1, 1]，1 表示完全一致，0 为随机水平。
- **同质性（Homogeneity）**：同一簇里的样本是否属于同一真实类别。
- **完整性（Completeness）**：同一真实类别的样本是否被分到同一簇。


In [ ]:
print(f"ARI         : {adjusted_rand_score(y, clusters):.3f}")
print(f"同质性 Homog: {homogeneity_score(y, clusters):.3f}")
print(f"完整性 Compl: {completeness_score(y, clusters):.3f}")


## PCA 降维可视化

把 27 维特征压缩到前两个主成分，在二维平面上观察：

- 左图按**真实类别**着色——检验有监督分类的难易度；
- 右图按**K-Means 簇**着色——检验无监督聚类是否还原了类似的结构。


In [ ]:
pca = PCA(n_components=2)   # 只保留前 2 个主成分
X_pca = pca.fit_transform(X_scaled)  # fit: 算协方差/做SVD，求出 v1、v2
                                    # transform: 完成投影，X_pca 是 1941×2
#PC1 占 30.8%，PC2 占 12.1%——意思是后 PC1 这一个轴承载了原始 27 维数据约三成的总方差。两个轴合计约 43%，也就是说降成二维时丢掉了约 57% 的信息，这是用低维可视化必然付出的代价。
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, coloring, title in [
    (axes[0], y, "按真实类别着色"),
    (axes[1], clusters, "按 K-Means 簇着色"),
]:
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=pd.factorize(coloring)[0],
                         s=8, alpha=0.6, cmap="tab10")
    ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
           ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", title=title)
plt.tight_layout()
plt.show()


## 三维 PCA 可视化

前两维只解释了约 43% 的方差，丢掉的信息不少。
再增加第三主成分 PC3，用 `mpl_toolkits.mplot3d` 在三维空间中观察——
在 Jupyter 中生成的三维图是**可交互的**（可用鼠标旋转视角）。
同样两张图对比：按真实类别着色 vs 按 K-Means 簇着色。


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  注册 3D 投影

pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)
print("前三个主成分的方差贡献率:",
      [f"{r:.1%}" for r in pca3.explained_variance_ratio_],
      "累计:", f"{pca3.explained_variance_ratio_.sum():.1%}")

fig = plt.figure(figsize=(13, 6))
for i, (coloring, title) in enumerate([(y, "按真实类别着色"),
                                       (clusters, "按 K-Means 簇着色")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2],
               c=pd.factorize(coloring)[0], s=8, alpha=0.6, cmap="tab10")
    ax.set(xlabel="PC1", ylabel="PC2", zlabel="PC3", title=title)
plt.tight_layout()
plt.show()


In [ ]:
# 簇画像：每个簇由哪些真实类别构成 + 关键特征均值
tab = pd.crosstab(clusters, y)
for k in range(7):
    n = tab.loc[k].sum()
    if n == 0:
        print(f"簇{k}: 空簇")
        continue
    top = tab.loc[k].idxmax()
    print(f"簇{k}: 主导={top}, 纯度={tab.loc[k].max()/n:.0%}, n={n}")

keys = ["Pixels_Areas", "Steel_Plate_Thickness", "TypeOfSteel_A300",
        "Sum_of_Luminosity", "Edges_Index", "Orientation_Index",
        "Outside_Global_Index", "Luminosity_Index", "SigmoidOfAreas"]
df.assign(簇=clusters).groupby("簇")[keys].mean().round(3)


## 每个簇代表什么质量模式

把簇的构成（主导类别）和特征均值结合起来，K-Means 找到的其实不是 7 种“缺陷类型”，
而是 7 种**质量模式**——按缺陷形态、位置和母材特征分组：

| 簇 | 规模 | 主导类别 | 特征画像 | 模式解读 |
|---|---|---|---|---|
| 0 | 126 | K_Scatch 96% | 面积极大(≈1.4万像素)、薄板 | 薄板上的大面积划痕 |
| 1 | 597 | Bumps 41%（混杂） | 面积小、板厚≈89、99% 为 A300 钢 | 厚板 A300 上的小斑点类缺陷 |
| 2 | 339 | Other_Faults 48%，**含 97% 的 Stains** | 面积小、亮度指数最高、Edges_Index 最高 | 小而亮、边缘锐利的污渍/混合缺陷 |
| 3 | 358 | Other_Faults 44%，含 Z_Scratch/Pastry | 板厚≈113、方向指数≈+0.66、贴边指数≈1.0 | 厚板上定向、延伸到板边的长条缺陷 |
| 4 | 290 | Other_Faults 36%，**含 48% 的 Pastry** | 中等面积、定向≈+0.50、贴边≈0.95 | 中等大小、定向、近板边的缺陷 |
| 5 | 1 | K_Scatch（单样本） | 面积 15 万像素（离群值） | 离群点独占一簇 |
| 6 | 230 | K_Scatch 86% | 面积大、薄板、方向≈-0.62、贴边≈0 | 薄板上完全内部、反向定向的大划痕 |

三个有洞察的发现：

1. **K_Scatch 被拆成多个“子模式”**：簇 0 是薄板超大型划痕，簇 6 是板内反向定向划痕。
   K-Means 认为形态差异比“同属一种缺陷”的标签更重要——这是聚类视角对人工分类的**细化**。
2. **Stains 被“重新发现”**：簇 2 囊括了 97% 的 Stains。污渍的特征指纹很清晰：
   面积小、亮度高、边缘锐利，K-Means 在没有标签的情况下捕捉到了这个物理类别。
3. **Bumps/Other_Faults/Pastry 按板材与位置而非形态被区分**：这三类在特征空间
   高度重叠（与有监督 notebook 的混淆矩阵一致），K-Means 转而按板厚、定向性、
   是否贴边切分它们。

## 小结

- k 的选择：轮廓系数在 k=2 时最高（≈0.30），肘部法无明显拐点——数据最自然的
  粗粒度结构是“两大群”，强行分成 7 簇反而制造了不清晰的结构。
- ARI 仅 0.156，聚类与真实类别对应关系弱，但上面的簇画像表明 K-Means 还原的是
  **真实的物理结构**（划痕子模式、污渍指纹、板材/位置维度），只是和人工标签
  不是一一对应关系——这正是 ARI 低的真正原因。
- 有监督 vs 无监督：KNN 靠标签能达到 0.78 的 macro F1，K-Means 不依赖标签只能
  得到 ARI 0.16。类别边界在特征空间中重叠严重，“标准答案”提供了巨大信息增益。
- 工程上两者互补：无监督用于探索结构、辅助标注；有监督用于标注充足时的准确预测。
